In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

print("Libraries imported successfully.")

Libraries imported successfully.


In [9]:
file_path = "salary_synthetic.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print("Dataset Shape:", df.shape)

df.head()

Dataset loaded successfully.
Dataset Shape: (2000, 15)


,age,gender,education,experience_years,role_seniority,company_size,location_tier,skills_count,certifications,worked_remote,last_promotion_years_ago,salary_bdt,recent_project_description_length,survey_date,recent_note
0,33,Male,M.Sc,6.0,Senior,Enterprise,Tier-2,1.0,2.0,1,0.0,145185,57.0,2024-11-15,Worked on microservices deployment
1,29,Male,M.Sc,9.0,Junior,Enterprise,Remote,5.0,0.0,1,0.0,121262,50.0,2024-03-08,Built ML pipeline for preprocessing
2,34,Male,B.Sc+Cert,NaN,Lead,Enterprise,Remote,5.0,1.0,1,5.0,184875,50.0,2024-09-01,Experience in data cleaning and ETL
3,39,Male,B.Sc,16.0,Mid,Enterprise,Tier-1,5.0,0.0,1,3.0,180105,59.0,2024-01-26,Contributed to open-source NLP repo
4,29,Female,B.Sc,8.0,Junior,SME,Tier-1,5.0,2.0,0,7.0,114750,NaN,2023-12-08,Built ML pipeline for preprocessing


In [10]:
df_original = df.copy()

print("Original dataset backup created.")

Original dataset backup created.


In [11]:
print("Data Types Before Correction:")
print(df.dtypes)

Data Types Before Correction:
age                                    int64
gender                                   str
education                                str
experience_years                     float64
role_seniority                           str
company_size                             str
location_tier                            str
skills_count                         float64
certifications                       float64
worked_remote                          int64
last_promotion_years_ago             float64
salary_bdt                             int64
recent_project_description_length    float64
survey_date                              str
recent_note                              str
dtype: object


In [13]:
numerical_columns = df.select_dtypes(
    include=["number"]
).columns.tolist()

text_columns = df.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Numerical Columns:")
print(numerical_columns)

print("\nText/Categorical Columns:")
print(text_columns)

Numerical Columns:
['age', 'experience_years', 'skills_count', 'certifications', 'worked_remote', 'last_promotion_years_ago', 'salary_bdt', 'recent_project_description_length']

Text/Categorical Columns:
['gender', 'education', 'role_seniority', 'company_size', 'location_tier', 'survey_date', 'recent_note']


In [18]:
# Data types before infer_objects()
dtypes_before = df.dtypes.astype(str)

# Automatically infer better data types
df = df.infer_objects()

# Data types after infer_objects()
dtypes_after = df.dtypes.astype(str)

# Comparison table
datatype_changes = pd.DataFrame({
    "Before": dtypes_before,
    "After": dtypes_after
})

# Show only changed data types
changed_columns = datatype_changes[
    datatype_changes["Before"] != datatype_changes["After"]
]

print("Changes made by infer_objects():")

if changed_columns.empty:
    print("No automatic datatype changes were required.")
else:
    display(changed_columns)

Changes made by infer_objects():
No automatic datatype changes were required.


In [17]:
possible_date_columns = [
    column for column in df.columns
    if (
        "date" in column.lower()
        or "time" in column.lower()
        or "timestamp" in column.lower()
    )
]

print("Automatically detected possible date columns:")
print(possible_date_columns)

Automatically detected possible date columns:
['survey_date']


In [19]:
date_validation_report = []

for column in possible_date_columns:
    
    converted_dates = pd.to_datetime(
        df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )
    
    invalid_count = (
        df[column].notna() & converted_dates.isna()
    ).sum()
    
    date_validation_report.append({
        "Column": column,
        "Invalid_Dates": invalid_count
    })

date_validation_report = pd.DataFrame(
    date_validation_report
)

print("Date Validation Report:")
display(date_validation_report)

Date Validation Report:


,Column,Invalid_Dates
0,survey_date,0


In [20]:
for column in possible_date_columns:
    
    df[column] = pd.to_datetime(
        df[column],
        format="%Y-%m-%d",
        errors="coerce"
    )

print("Date columns converted successfully.")

print("\nsurvey_date datatype:")
print(df["survey_date"].dtype)

print("\nDate range:")
print("Minimum Date:", df["survey_date"].min())
print("Maximum Date:", df["survey_date"].max())

Date columns converted successfully.

survey_date datatype:
datetime64[us]

Date range:
Minimum Date: 2023-01-01 00:00:00
Maximum Date: 2024-12-30 00:00:00


In [21]:
categorical_columns = df.select_dtypes(
    include=["object", "str"]
).columns.tolist()

print("Categorical Columns:")
print(categorical_columns)

Categorical Columns:
['gender', 'education', 'role_seniority', 'company_size', 'location_tier', 'recent_note']


In [22]:
for column in categorical_columns:
    
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print("Categorical values standardized successfully.")

Categorical values standardized successfully.


In [24]:
for column in categorical_columns:
    print(df[column].value_counts(dropna=False))

gender
Male      1437
Female     530
Other       33
Name: count, dtype: int64[pyarrow]
education
B.Sc         1070
M.Sc          445
B.Sc+Cert     229
<NA>          100
M.Eng          99
PhD            57
Name: count, dtype: int64[pyarrow]
role_seniority
Mid       791
Junior    715
Senior    393
Lead      101
Name: count, dtype: int64[pyarrow]
company_size
SME           810
Startup       596
Enterprise    594
Name: count, dtype: int64[pyarrow]
location_tier
Tier-1    813
Tier-2    620
Remote    321
Tier-3    246
Name: count, dtype: int64[pyarrow]
recent_note
Built ML pipeline for preprocessing    438
No relevant project mentioned          354
Contributed to open-source NLP repo    332
Worked on microservices deployment     323
<NA>                                   300
Experience in data cleaning and ETL    253
Name: count, dtype: int64[pyarrow]


In [25]:
category_summary = []

for column in categorical_columns:
    
    category_summary.append({
        "Column": column,
        "Unique_Values": df[column].nunique(dropna=True),
        "Missing_Values": df[column].isna().sum()
    })

category_summary = pd.DataFrame(category_summary)

print("Categorical Column Summary:")
display(category_summary)

Categorical Column Summary:


,Column,Unique_Values,Missing_Values
0,gender,3,0
1,education,5,100
2,role_seniority,4,0
3,company_size,3,0
4,location_tier,4,0
5,recent_note,5,300


In [26]:
df["survey_year"] = df["survey_date"].dt.year
df["survey_month"] = df["survey_date"].dt.month
df["survey_day"] = df["survey_date"].dt.day

print("Date features created successfully.")

display(
    df[[
        "survey_date",
        "survey_year",
        "survey_month",
        "survey_day"
    ]].head()
)

Date features created successfully.


,survey_date,survey_year,survey_month,survey_day
0,2024-11-15,2024,11,15
1,2024-03-08,2024,3,8
2,2024-09-01,2024,9,1
3,2024-01-26,2024,1,26
4,2023-12-08,2023,12,8


In [27]:
df_encoded = df.copy()

# Original date is no longer required because date features exist
df_encoded = df_encoded.drop(
    columns=["survey_date"]
)

print("Separate encoding dataset created.")
print("Shape before encoding:", df_encoded.shape)

Separate encoding dataset created.
Shape before encoding: (2000, 17)


In [29]:
seniority_mapping = {
    "Junior": 0,
    "Mid": 1,
    "Senior": 2,
    "Lead": 3
}

df_encoded["role_seniority_encoded"] = (
    df_encoded["role_seniority"]
    .map(seniority_mapping)
    .astype("Int64")
)

print("Seniority encoding:")

display(
    df_encoded[[
        "role_seniority",
        "role_seniority_encoded"
    ]]
    .drop_duplicates()
    .sort_values("role_seniority_encoded")
)

Seniority encoding:


,role_seniority,role_seniority_encoded
1,Junior,0
3,Mid,1
0,Senior,2
2,Lead,3


In [30]:
df_encoded = df_encoded.drop(
    columns=["role_seniority"]
)

print("Original role_seniority column removed.")

Original role_seniority column removed.


In [31]:
nominal_columns = [
    "gender",
    "education",
    "company_size",
    "location_tier",
    "recent_note"
]

df_encoded = pd.get_dummies(
    df_encoded,
    columns=nominal_columns,
    prefix=nominal_columns,
    dummy_na=True,
    dtype=int
)

print("One-hot encoding completed.")
print("Encoded Dataset Shape:", df_encoded.shape)

One-hot encoding completed.
Encoded Dataset Shape: (2000, 37)


In [32]:
print("Encoded Dataset Columns:")

for number, column in enumerate(
    df_encoded.columns,
    start=1
):
    print(number, column)

Encoded Dataset Columns:
1 age
2 experience_years
3 skills_count
4 certifications
5 worked_remote
6 last_promotion_years_ago
7 salary_bdt
8 recent_project_description_length
9 survey_year
10 survey_month
11 survey_day
12 role_seniority_encoded
13 gender_Female
14 gender_Male
15 gender_Other
16 gender_<NA>
17 education_B.Sc
18 education_B.Sc+Cert
19 education_M.Eng
20 education_M.Sc
21 education_PhD
22 education_<NA>
23 company_size_Enterprise
24 company_size_SME
25 company_size_Startup
26 company_size_<NA>
27 location_tier_Remote
28 location_tier_Tier-1
29 location_tier_Tier-2
30 location_tier_Tier-3
31 location_tier_<NA>
32 recent_note_Built ML pipeline for preprocessing
33 recent_note_Contributed to open-source NLP repo
34 recent_note_Experience in data cleaning and ETL
35 recent_note_No relevant project mentioned
36 recent_note_Worked on microservices deployment
37 recent_note_<NA>


In [34]:
remaining_text_columns = (
    df_encoded
    .select_dtypes(include=["object", "str"])
    .columns
    .tolist()
)

print("Remaining Text Columns:")
print(remaining_text_columns)

Remaining Text Columns:
[]


In [35]:
print("Encoded Dataset Data Types:")
print(df_encoded.dtypes)

Encoded Dataset Data Types:
age                                                  int64
experience_years                                   float64
skills_count                                       float64
certifications                                     float64
worked_remote                                        int64
last_promotion_years_ago                           float64
salary_bdt                                           int64
recent_project_description_length                  float64
survey_year                                          int32
survey_month                                         int32
survey_day                                           int32
role_seniority_encoded                               Int64
gender_Female                                        int64
gender_Male                                          int64
gender_Other                                         int64
gender_<NA>                                          int64
education_B.Sc              

In [36]:
missing_values = df.isna().sum()

missing_report = pd.DataFrame({
    "Missing_Count": missing_values,
    "Missing_Percentage": (
        missing_values / len(df) * 100
    ).round(2)
})

missing_report = missing_report[
    missing_report["Missing_Count"] > 0
]

print("Missing Values After Task 7:")
display(missing_report)

print(
    "Total Missing Values:",
    df.isna().sum().sum()
)

Missing Values After Task 7:


,Missing_Count,Missing_Percentage
education,100,5.0
experience_years,160,8.0
skills_count,200,10.0
certifications,80,4.0
last_promotion_years_ago,120,6.0
recent_project_description_length,240,12.0
recent_note,300,15.0


Total Missing Values: 1200


In [37]:
datatype_comparison = pd.DataFrame({
    "Original_Datatype": df_original.dtypes.astype(str),
    "Corrected_Datatype": df[
        df_original.columns
    ].dtypes.astype(str)
})

print("Original vs Corrected Data Types:")
display(datatype_comparison)

Original vs Corrected Data Types:


,Original_Datatype,Corrected_Datatype
age,int64,int64
gender,str,string
education,str,string
experience_years,float64,float64
role_seniority,str,string
company_size,str,string
location_tier,str,string
skills_count,float64,float64
certifications,float64,float64
worked_remote,int64,int64


In [39]:
print("Original Dataset Shape:", df_original.shape)
print("Standardized Dataset Shape:", df.shape)
print("Encoded Dataset Shape:", df_encoded.shape)

print("\nDuplicate Rows:", df.duplicated().sum())

print(
    "Invalid or Missing Survey Dates:",
    df["survey_date"].isna().sum()
)

print(
    "Survey Date Datatype:",
    df["survey_date"].dtype
)

print(
    "Remaining Text Columns in Encoded Dataset:",
    len(
        df_encoded.select_dtypes(
            include=["object", "str"]
        ).columns
    )
)

print(
    "Missing Values Preserved for Task 9:",
    df.isna().sum().sum()
)

Original Dataset Shape: (2000, 15)
Standardized Dataset Shape: (2000, 18)
Encoded Dataset Shape: (2000, 37)

Duplicate Rows: 0
Invalid or Missing Survey Dates: 0
Survey Date Datatype: datetime64[us]
Remaining Text Columns in Encoded Dataset: 0
Missing Values Preserved for Task 9: 1200


In [40]:
df.head()

,age,gender,education,experience_years,role_seniority,company_size,location_tier,skills_count,certifications,worked_remote,last_promotion_years_ago,salary_bdt,recent_project_description_length,survey_date,recent_note,survey_year,survey_month,survey_day
0,33,Male,M.Sc,6.0,Senior,Enterprise,Tier-2,1.0,2.0,1,0.0,145185,57.0,2024-11-15,Worked on microservices deployment,2024,11,15
1,29,Male,M.Sc,9.0,Junior,Enterprise,Remote,5.0,0.0,1,0.0,121262,50.0,2024-03-08,Built ML pipeline for preprocessing,2024,3,8
2,34,Male,B.Sc+Cert,NaN,Lead,Enterprise,Remote,5.0,1.0,1,5.0,184875,50.0,2024-09-01,Experience in data cleaning and ETL,2024,9,1
3,39,Male,B.Sc,16.0,Mid,Enterprise,Tier-1,5.0,0.0,1,3.0,180105,59.0,2024-01-26,Contributed to open-source NLP repo,2024,1,26
4,29,Female,B.Sc,8.0,Junior,SME,Tier-1,5.0,2.0,0,7.0,114750,NaN,2023-12-08,Built ML pipeline for preprocessing,2023,12,8


In [41]:
df_encoded.head()

,age,experience_years,skills_count,certifications,worked_remote,last_promotion_years_ago,salary_bdt,recent_project_description_length,survey_year,survey_month,survey_day,role_seniority_encoded,gender_Female,gender_Male,gender_Other,gender_<NA>,education_B.Sc,education_B.Sc+Cert,education_M.Eng,education_M.Sc,education_PhD,education_<NA>,company_size_Enterprise,company_size_SME,company_size_Startup,company_size_<NA>,location_tier_Remote,location_tier_Tier-1,location_tier_Tier-2,location_tier_Tier-3,location_tier_<NA>,recent_note_Built ML pipeline for preprocessing,recent_note_Contributed to open-source NLP repo,recent_note_Experience in data cleaning and ETL,recent_note_No relevant project mentioned,recent_note_Worked on microservices deployment,recent_note_<NA>
0,33,6.0,1.0,2.0,1,0.0,145185,57.0,2024,11,15,2,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0
1,29,9.0,5.0,0.0,1,0.0,121262,50.0,2024,3,8,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0
2,34,NaN,5.0,1.0,1,5.0,184875,50.0,2024,9,1,3,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0
3,39,16.0,5.0,0.0,1,3.0,180105,59.0,2024,1,26,1,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0
4,29,8.0,5.0,2.0,0,7.0,114750,NaN,2023,12,8,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0
